# Score your tracker offline with the OFFICIAL metric

Competition kernels have no internet, so you cannot `git clone` or `pip install`
the organisers' scoring package. This notebook attaches it as a dataset and shows
how to **score a submission locally before spending a daily submission slot**.

It also builds intuition for the metric the only way that really works: take a
*perfect* prediction, break it in one specific way at a time, and watch what the
score does.

**No GPU, no model, no training.** Everything below runs on the ground-truth
graphs in a couple of minutes.

## Two things that surprised me

1. **`adj_edge_jaccard` can exceed 1.0.** The node-count adjustment is
   `J * (1 - 0.1 * (N_pred - N_true) / N_true)`, so *under*-predicting nodes
   multiplies your Jaccard by more than one.
2. **Edge false positives and false negatives are coupled.** A mis-link creates
   one of each, so fixing a single mis-link is worth roughly twice as much as
   recovering a missed edge.

Dataset used: `biohub-official-scorer-patched` - an unmodified copy of
[royerlab/kaggle-cell-tracking-competition](https://github.com/royerlab/kaggle-cell-tracking-competition)
pinned at commit `075fc5f`, which includes the 2026-07-17 division-metric patch.
All credit to the organisers; the dataset is redistribution for offline use only.


## 0. Install the offline dependencies

`tracksdata`, `geff`, `zarr` and friends are not preinstalled on Kaggle and there
is no internet, so we install them from the wheels bundled in the public
`biohub-tracking-support-pack-50ep-v1` dataset.

**Note the subtlety.** That same pack also vendors its own copy of the scoring
code as the module `biohub_tracking` - and that copy predates the 2026-07-17
division patch. We use the pack **only for the wheels**, and import the metric
itself from the pinned patched copy. Cell 3 asserts we got the right one. This is
exactly the trap worth avoiding: if you already have this pack attached and do
`from biohub_tracking.metrics import evaluate`, you are scoring divisions with the
pre-patch rule.


In [ ]:
try:
    import zarr, geff, tracksdata
    
except: 
    import os
    import sys
    import subprocess
    import importlib
    import importlib.util
    from pathlib import Path
    
    # Polars wheel in the support package uses the 32-bit runtime package.
    os.environ.setdefault("POLARS_PREFER_PKG", "32")
    
    SUPPORT_DIR = Path(
        "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"
    )
    WHEELS_DIR = SUPPORT_DIR / "wheels"
    
    if not WHEELS_DIR.exists():
        # Fallback for Kaggle's shorter mounted dataset path.
        candidates = list(Path("/kaggle/input").glob("**/wheels"))
        if not candidates:
            raise FileNotFoundError("Could not find the attached offline wheels directory")
        WHEELS_DIR = candidates[0]
    
    print("Offline wheels:", WHEELS_DIR)
    
    
    # These are packages supplied by the attached dataset.
    # Deliberately exclude:
    #   numpy, scipy, torch, pandas, dask, xarray, numba, llvmlite
    #
    # Kaggle already supplies compatible versions of those packages.
    OFFLINE_PACKAGES = [
        "tracksdata",
        "zarr==3.2.1",
        "numcodecs==0.15.1",
        "donfig==0.8.1.post1",
        "geff==1.2.0.1.1",
        "geff-spec==1.1.1",
        "pyscipopt==6.2.1",
        "ilpy==0.6.0",
        "rustworkx==0.18.0",
        "polars==1.42.0",
        "polars-runtime-32==1.42.0",
        "bidict==0.23.1",
        "imagecodecs==2026.6.26",
    ]
    
    
    def module_missing(module_name: str) -> bool:
        return importlib.util.find_spec(module_name) is None
    
    
    # Import name differs from pip package name in several cases.
    REQUIRED_IMPORTS = {
        "tracksdata": "tracksdata",
        "zarr": "zarr",
        "numcodecs": "numcodecs",
        "geff": "geff",
        "pyscipopt": "pyscipopt",
        "ilpy": "ilpy",
        "rustworkx": "rustworkx",
        "polars": "polars",
        "imagecodecs": "imagecodecs",
    }
    
    
    def purge_modules(module_roots):
        """
        Remove already-imported package modules from sys.modules.
    
        Normally this cell runs before imports, but this also protects against
        accidental imports performed by earlier Kaggle initialization code.
        """
        for root in module_roots:
            for name in list(sys.modules):
                if name == root or name.startswith(root + "."):
                    sys.modules.pop(name, None)
    
    
    def install_offline_packages():
        cmd = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-index",
            "--no-deps",
            "--find-links",
            str(WHEELS_DIR),
            *OFFLINE_PACKAGES,
        ]
    
        print("Installing attached packages without modifying NumPy/SciPy/Torch...")
        result = subprocess.run(
            cmd,
            text=True,
            capture_output=True,
        )
    
        if result.returncode != 0:
            print(result.stdout[-4000:])
            print(result.stderr[-4000:])
            raise RuntimeError("Offline dependency installation failed")
    
        purge_modules(REQUIRED_IMPORTS.values())
    
    
    install_offline_packages()
    
    
    # Verify the complete import chain needed by biohub_tracking.io.
    failures = {}
    
    for name, module_name in {
        **REQUIRED_IMPORTS,
        "numpy": "numpy",
        "scipy": "scipy",
        "dask": "dask.array",
        "xarray": "xarray",
    }.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    
    if failures:
        raise ImportError(
            "Dependency verification failed:\n"
            + "\n".join(f"{name}: {error}" for name, error in failures.items())
        )

import zarr, geff, tracksdata
print("All offline dependencies imported successfully.")

## 1. Load the scorer

Kaggle mounts private datasets under `/kaggle/input/datasets/<owner>/<slug>/` and
public ones under `/kaggle/input/<slug>/`, so search for the package instead of
hard-coding a path.

In [ ]:
import sys, os, glob, json
import numpy as np

cands = []
for pat in ("/kaggle/input/*/*/*/src/tracking_cellmot/metrics.py",
            "/kaggle/input/*/*/src/tracking_cellmot/metrics.py",
            "/kaggle/input/**/tracking_cellmot/metrics.py"):
    cands += glob.glob(pat, recursive=True)
assert cands, "scorer dataset not attached"
SRC  = os.path.dirname(os.path.dirname(cands[0]))
REPO = os.path.dirname(SRC)
sys.path.insert(0, SRC)
print("package:", SRC)

import tracksdata as td
import polars as pl
from geff import GeffMetadata
from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import evaluate, node_recall, per_sample_metrics
print("imported OK")


### Verify you loaded the PATCHED metric

Worth doing explicitly. Some widely used support packs vendor a copy of this code
that predates the 2026-07-17 patch and still scores divisions with the old
weakly-connected-component rule. Silently scoring with the wrong metric is easy to
miss, so assert rather than assume.

In [ ]:
import inspect
import tracking_cellmot.division_metrics as dm
src = inspect.getsource(dm)
assert "_weakly_connected_components" not in src, "PRE-PATCH metric loaded!"
print("patched metric confirmed ('_weakly_connected_components' absent)")


## 2. Load one labelled training movie

`open_dataset(..., load_image=False)` reads only the tracking graph, so this is
fast. `estimated_number_of_nodes` from the GEFF metadata is `N_true` for the
node-count adjustment - note it counts **all** cells, including unannotated ones,
not just the labelled subset.

In [ ]:
COMP = "/kaggle/input/competitions/biohub-cell-tracking-during-development"
if not os.path.exists(COMP):
    COMP = glob.glob("/kaggle/input/*biohub-cell-tracking*")[0]
TRAIN = f"{COMP}/train"

stems = sorted(p[:-5] for p in os.listdir(TRAIN) if p.endswith(".zarr")
               and os.path.exists(os.path.join(TRAIN, p[:-5] + ".geff")))
print(f"{len(stems)} labelled train movies")

# Annotation density varies enormously between movies - some carry ~50 GT edges,
# others over a thousand. Pick the most densely annotated one, otherwise the
# perturbation demos below have too little to work with.
counts = {}
for s in stems:
    try:
        counts[s] = GeffMetadata.read(f"{TRAIN}/{s}.geff").extra.get("estimated_number_of_nodes", 0)
    except Exception:
        pass
probe = sorted(stems, key=lambda s: -float(counts.get(s, 0)))[:12]
best, best_e = None, -1
for s in probe:
    try:
        d = open_dataset(f"{TRAIN}/{s}.zarr", normalize=False, load_image=False, require_tracks=True)
        if d.tracks.num_edges() > best_e:
            best, best_e = s, d.tracks.num_edges()
    except Exception:
        continue
SAMPLE = best
print(f"using {SAMPLE} ({best_e:,} annotated edges - the densest of {len(probe)} probed)")

ds   = open_dataset(f"{TRAIN}/{SAMPLE}.zarr", normalize=False, load_image=False, require_tracks=True)
gt   = ds.tracks
meta = GeffMetadata.read(f"{TRAIN}/{SAMPLE}.geff")
N_TRUE = float(meta.extra["estimated_number_of_nodes"])

print(f"GT nodes            : {gt.num_nodes():,}")
print(f"GT edges            : {gt.num_edges():,}")
print(f"voxel scale (z,y,x) : {ds.scale}")
print(f"N_true (all cells)  : {N_TRUE:,.0f}")
print(f"labelled fraction   : {gt.num_nodes()/N_TRUE:.2%}  <- labels are SPARSE")


## 3. Rebuild the GT as a "prediction"

`ILPSolver.solve()` returns a `GraphView`, and `evaluate()` internally calls
`.copy()`, which `GraphView` does not support - so building a fresh
`InMemoryGraph` is the safe pattern (`.detach()` also works).

In [ ]:
gt_nodes = list(gt.node_attrs().iter_rows(named=True))
gt_edges = list(gt.edge_attrs().iter_rows(named=True))
gt_id    = {int(r["node_id"]): r for r in gt_nodes}

def build(nodes, edges):
    """nodes: list of dicts with t,z,y,x (+node_id). edges: (src,tgt) node_id pairs."""
    g = td.graph.InMemoryGraph()
    for k in ("z", "y", "x"):
        g.add_node_attr_key(k, pl.Float64, -999999.0)
    ids = [int(n["node_id"]) for n in nodes]
    new = g.bulk_add_nodes([{"t": int(n["t"]), "z": float(n["z"]),
                             "y": float(n["y"]), "x": float(n["x"])} for n in nodes])
    remap = dict(zip(ids, new))
    keep = [(s, t) for s, t in edges if s in remap and t in remap]
    if keep:
        g.bulk_add_edges([{"source_id": remap[s], "target_id": remap[t]} for s, t in keep])
    return g

def score(nodes, edges, label):
    g  = build(nodes, edges)
    er = evaluate(g, gt, scale=ds.scale, max_distance=7.0)
    m  = per_sample_metrics(er=er, n_total=N_TRUE, node_recall=node_recall(g, gt))
    row = dict(variant=label, nodes=len(nodes), edges=g.num_edges(),
               eTP=er.edge_tp, eFP=er.edge_fp, eFN=er.edge_fn,
               J=m["edge_jaccard"], adjJ=m["adj_edge_jaccard"])
    print(f"{label:<34} nodes={len(nodes):>7,} TP/FP/FN={er.edge_tp:>5}/{er.edge_fp:>4}/{er.edge_fn:>4} "
          f"J={m['edge_jaccard']:.4f}  adjJ={m['adj_edge_jaccard']:.4f}")
    return row

edge_pairs = [(int(e["source_id"]), int(e["target_id"])) for e in gt_edges]
results = [score(gt_nodes, edge_pairs, "perfect (GT as prediction)")]


## 4. Break it, one thing at a time

Each variant changes exactly one property, so the score movement is attributable.

In [ ]:
rng = np.random.default_rng(0)

# (a) drop 10% of edges -> pure false negatives, no new false positives
keep = rng.permutation(len(edge_pairs))[: int(0.9 * len(edge_pairs))]
results.append(score(gt_nodes, [edge_pairs[i] for i in sorted(keep)], "(a) 10% of edges dropped"))

# (b) rewire 10% of edges to a random target in the next frame -> FP *and* FN together
by_t = {}
for r in gt_nodes:
    by_t.setdefault(int(r["t"]), []).append(int(r["node_id"]))
rewired, n_rw = [], 0
for s, t in edge_pairs:
    if rng.random() < 0.10:
        pool = by_t.get(int(gt_id[t]["t"]), [])
        if len(pool) > 1:
            alt = int(rng.choice(pool))
            if alt != t:
                rewired.append((s, alt)); n_rw += 1; continue
    rewired.append((s, t))
results.append(score(gt_nodes, rewired, f"(b) {n_rw} edges rewired"))


### The node-count adjustment, in both directions

This is the part most people miss. `adj_edge_jaccard = J * (1 - 0.1 * (N_pred - N_true)/N_true)`
uses `N_true` = **all** cells, not the labelled ones. Since the labelled subset is
a small fraction of the movie, submitting only the GT nodes already
*under*-predicts massively - which is why the "perfect" row above can show
`adjJ > J`.

In [ ]:
# (c) add spurious isolated nodes -> pushes N_pred up toward / past N_true
for extra in (int(0.5 * N_TRUE), int(1.5 * N_TRUE)):
    pad = [{"node_id": 10_000_000 + i, "t": int(rng.integers(0, 100)),
            "z": float(rng.integers(0, 50)), "y": float(rng.integers(0, 2000)),
            "x": float(rng.integers(0, 2000))} for i in range(extra)]
    results.append(score(gt_nodes + pad, edge_pairs, f"(c) +{extra:,} spurious nodes"))

import pandas as pd
df = pd.DataFrame(results)
df["node_ratio"] = (df.nodes - N_TRUE) / N_TRUE
df["adj_multiplier"] = df.adjJ / df.J.replace(0, np.nan)
print()
print(df[["variant","nodes","node_ratio","J","adjJ","adj_multiplier"]].to_string(index=False))


## 5. What this tells you

- **(a) vs (b)**: dropping edges costs false negatives only; *rewiring* the same
  number costs a false positive **and** a false negative. Mis-links are the
  expensive error - fixing one is worth about twice recovering a missed edge.
- **(c)**: the adjustment multiplier tracks `1 - 0.1 * node_ratio`. Under-predict
  and it exceeds 1; over-predict and it bites. It is a real, tunable term.
- **Sparse labels**: edges whose endpoints match no annotated GT node are ignored
  entirely. That makes a local score a good *diagnostic* but an unreliable
  *predictor* - in my own testing, the local ranking of two pipelines came out
  inverted relative to the public leaderboard. Use it to catch invalid graphs and
  crashes, not to choose between close candidates.

## 6. The documented round trip: submission CSV -> score

The repo ships `csv_to_geffs.py` and `evaluate.py`, so you can score an existing
`submission.csv` directly.

In [ ]:
import csv, subprocess
OUT = "/kaggle/working/demo_submission.csv"
COLS = ["id","dataset","row_type","node_id","t","z","y","x","source_id","target_id"]
with open(OUT, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=COLS); w.writeheader()
    rid = 0
    for n in gt_nodes:
        w.writerow({"id": rid, "dataset": SAMPLE, "row_type": "node",
                    "node_id": int(n["node_id"]), "t": int(n["t"]),
                    "z": max(0, int(round(float(n["z"])))), "y": max(0, int(round(float(n["y"])))),
                    "x": max(0, int(round(float(n["x"])))), "source_id": -1, "target_id": -1}); rid += 1
    for s, t in edge_pairs:
        w.writerow({"id": rid, "dataset": SAMPLE, "row_type": "edge", "node_id": -1,
                    "t": -1, "z": -1, "y": -1, "x": -1, "source_id": s, "target_id": t}); rid += 1
print(f"wrote {rid:,} rows")

env = dict(os.environ); env["PYTHONPATH"] = SRC
for cmd in ([sys.executable, f"{REPO}/scripts/csv_to_geffs.py", "--csv", OUT, "--out-dir", "/kaggle/working/pred_geffs"],
            [sys.executable, f"{REPO}/scripts/evaluate.py", "--pred-dir", "/kaggle/working/pred_geffs", "--gt-dir", TRAIN]):
    p = subprocess.run(cmd, capture_output=True, text=True, env=env)
    print("\n$", " ".join(os.path.basename(c) if "/" in c else c for c in cmd))
    print(p.stdout[-2500:] or p.stderr[-2500:])


## Gotchas worth knowing

- `ILPSolver.solve()` returns a `GraphView`; `evaluate()` calls `.copy()` on the
  prediction, which `GraphView` rejects. Call `.detach()` or rebuild the graph.
- Aggregation across datasets is not a plain mean: `adj_edge_jaccard` is
  weight-averaged by `TP + FP + FN`, while `division_jaccard` is pooled (micro)
  across datasets and turned into a single Jaccard afterwards. Movies with few
  labelled edges barely count.
- `node_id` is **dataset-local**. Keying edges or degrees by a global `node_id`
  produces false errors, because ids repeat across datasets.
- `division_jaccard` is `nan` when a movie contains no annotated divisions (0/0/0), so guard before averaging.
- Check you loaded the patched metric. A stale vendored copy will happily
  score divisions with the pre-patch rule.

If this saved you a submission slot, an upvote on the
[dataset](https://www.kaggle.com/datasets/dalloliogm/biohub-official-scorer-patched)
is appreciated. Credit for the scoring code belongs to the competition organisers.